### Refactoring Demo

Stage 15 (Orchestration and System Design)

In [1]:
import argparse
import json
import logging
import sys
from datetime import datetime
from pathlib import Path
import pandas as pd
import yfinance as yf

def ingest_data(output_path: str, ticker: str = "SPY", period: str = "2y") -> None:
    '''Ingest stock data from yfinance API and save to parquet'''
    logging.info('[ingest_data] start for ticker %s', ticker)
    
    try:
        # Fetch data
        data = yf.download(ticker, period=period, progress=False)
        
        # Add metadata
        data['ingestion_timestamp'] = datetime.utcnow()
        data['ticker'] = ticker
        
        # Ensure output directory exists
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        
        # Save to parquet
        data.to_parquet(output_path)
        logging.info('[ingest_data] wrote %s with %d rows', output_path, len(data))
        
    except Exception as e:
        logging.error('[ingest_data] failed: %s', str(e))
        raise

def main(argv=None):
    parser = argparse.ArgumentParser(description='Data ingestion task for SPY ETF')
    parser.add_argument('--output', required=True, help='Output parquet file path')
    parser.add_argument('--ticker', default='SPY', help='Stock ticker symbol')
    parser.add_argument('--period', default='2y', help='Time period to download')
    
    args = parser.parse_args(argv)
    logging.basicConfig(level=logging.INFO, 
                       format='%(asctime)s - %(levelname)s - %(message)s',
                       handlers=[logging.StreamHandler(sys.stdout)])
    
    ingest_data(args.output, args.ticker, args.period)

if __name__ == '__main__':
    # Example CLI call within notebook for testing
    main(['--output', 'data/raw/spy_data.parquet', '--ticker', 'SPY', '--period', '2y'])

2025-08-27 13:37:40,449 - INFO - [ingest_data] start for ticker SPY


/var/folders/_4/t03mdfy94ts0ylt8cw0q1bp00000gn/T/ipykernel_36384/832384102.py:16: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period=period, progress=False)


2025-08-27 13:37:40,834 - INFO - [ingest_data] wrote data/raw/spy_data.parquet with 502 rows


In [2]:
import time
from typing import Callable

def retry(n_tries: int = 3, delay: float = 1.0, backoff: float = 2.0):
    """Retry decorator with exponential backoff"""
    def decorator(func: Callable):
        def wrapper(*args, **kwargs):
            attempts = 0
            current_delay = delay
            
            while attempts < n_tries:
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    attempts += 1
                    if attempts == n_tries:
                        logging.error('[retry] Failed after %d attempts: %s', n_tries, str(e))
                        raise
                    
                    logging.warning('[retry] Attempt %d failed: %s. Retrying in %.1fs', 
                                  attempts, str(e), current_delay)
                    time.sleep(current_delay)
                    current_delay *= backoff  # Exponential backoff
        return wrapper
    return decorator

# Example usage with retry
@retry(n_tries=3, delay=2.0, backoff=2.0)
def robust_ingest_data(output_path: str, ticker: str = "SPY", period: str = "2y") -> None:
    '''Ingest data with retry logic for transient failures'''
    ingest_data(output_path, ticker, period)